# Prepare the Data

**Goal:** turn the raw training table into a table of numbers that a model can learn from, and save the whole recipe as `preprocessor.pkl` so it can be applied to the train set, the test set, or a new sighting in exactly the same way.

**Ground rules from the checklist and the project**
- **Work on copies of the data.** Nothing below changes `train_set.parquet`.
- **Write functions for all transformations.** They live in `custom_transformers.py` (next to this notebook), so the Training notebook can import and tune them.
- **Training set only.** Every number the recipe learns (medians, means, categories, highlight colors) comes from the training set. The test set is **not opened** in this notebook.
- **No prepared data is saved.** Only `preprocessor.pkl` is created. The project says no data may be committed except the two parquet files.

| Checklist step | Section |
|---|---|
| 1. Data cleaning | 1 |
| 2. Feature selection | 2 |
| 3. Feature engineering | 3 |
| 4. Feature scaling (and encoding) | 4 |
| Project requirement: `preprocessor.pkl` | 6 |

Sections 5 and 6 list the settings that can be tuned later and the checks that the recipe is safe to use.

In [1]:
# Imports
from custom_transformers import *

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

In [2]:
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 90)

TARGET = "diseased"

## Load a copy of the training set

The label is kept apart from the features from the start, so it can never end up as an input column.

In [3]:
train = pd.read_parquet("train_set.parquet")     # written by the Get the Data step
work = train.copy()                               # every step below works on this copy
y_train = work[TARGET].astype(int)

print(f"{work.shape[0]:,} rows x {work.shape[1]} columns | diseased: {y_train.sum()} ({y_train.mean():.1%})")

2,418 rows x 49 columns | diseased: 303 (12.5%)


## A decision for every column

Each raw column gets one decision, and the reason comes from the Exploration notebook (training set only). The code cell after the table checks that **no column was forgotten**.

| Group | Columns | Decision | Reason (from Exploration) |
|---|---|---|---|
| Location | `X`, `Y` | **keep** | Weak south-to-north gradient. `Y` and `X` also carry what the hectare grid says |
| Fur and age | `Primary Fur Color`, `Highlight Fur Color`, `Age` | **keep, cleaned** | Primary fur color is by far the strongest signal (black 66%, cinnamon 49%, gray 4.5%) |
| Where it was | `Location`, `Above Ground Sighter Measurement` | **keep, cleaned** | Mixed text and numbers in the height column |
| Behavior | 13 tick boxes | **keep** | `Approaches` is the only clear one. The rest are kept because there are few of them |
| Time of day | `Shift` | **keep** | No signal found, but it is one cheap column. Tested in the Short-List step (Training 6a): removing it makes no clear difference for logistic regression, so it is kept |
| Named places | `geo_dist_water_m`, `geo_dist_green_m`, `geo_nearest_place_kind` | **keep** | The external geography source (distances in metres) |
| Duplicates | `Lat/Long`, `Date`, `Combination of Primary and Highlight Color`, `geo_dist_hill_m`, `geo_nearest_place` | **drop** | Same information as another column. `geo_dist_hill_m` has correlation -0.99 with `Y` |
| Identifier | `Unique Squirrel ID` | **drop** | Names a row, carries no general pattern |
| Hectare | `Hectare`, `Hectare Squirrel Number` | **drop** | 329 hectares with a median of 7 sightings each, and `X`/`Y` already say where. The squirrel number is a counting order given by the volunteers, which the app would not have when a report arrives |
| Free text (notes) | `Color notes`, `Other Activities`, `Other Interactions` | **optional**: keyword similarity (`use_keyword_similarity`) or spaCy meaning similarity (`use_spacy_similarity`), each with a "has a note" flag, both off by default | About 3 in 4 sightings have no note and the symptom words are not there, but whether a note exists may carry signal (Exploration, section 5). The Training step decides |
| Place text | `Specific Location` | **drop** | Describes a place, not the squirrel, and 84% is empty |
| People only | `geo_nearest_place_dist_m`, `geo_nearest_place_direction`, `geo_where_text` | **drop** | Built for animal control to read, not for the model |
| Day-level | `date`, `wx_*`, `cal_*` | **drop by default** | Only 11 distinct days, so each is really a "which day" code, and the random split puts the same days in train and test. Wind is also about half missing. Can be switched on (section 5) |
| Target | `diseased` | **target** | Kept apart as `y_train` |

In [4]:
DECISIONS = {
    # kept (cleaned or built into new columns by SquirrelFeatures)
    "X": "keep", "Y": "keep", "Shift": "keep", "Age": "keep, unknown -> 'Unknown'",
    "Primary Fur Color": "keep, missing -> 'Unknown'", "Highlight Fur Color": "keep, one 0/1 column per color",
    "Location": "keep, missing -> 'Unknown'", "Above Ground Sighter Measurement": "keep, text -> number + missing flag",
    **{b: "keep (0/1)" for b in BEHAVIORS},
    "geo_dist_water_m": "keep", "geo_dist_green_m": "keep", "geo_nearest_place_kind": "keep, one-hot",
    # dropped
    "Unique Squirrel ID": "drop: identifier", "Lat/Long": "drop: duplicate of X and Y", "Date": "drop: duplicate of date",
    "Combination of Primary and Highlight Color": "drop: duplicate of the two color columns",
    "geo_dist_hill_m": "drop: repeats Y (corr -0.99)", "geo_nearest_place": "drop: same as geo_nearest_place_kind",
    "Hectare": "drop: X and Y say where", "Hectare Squirrel Number": "drop: volunteer counting order, not known at report time",
    "Color notes": "keep as text: keyword / spaCy similarity (optional)",
    "Other Activities": "keep as text: keyword / spaCy similarity (optional)",
    "Other Interactions": "keep as text: keyword / spaCy similarity (optional)",
    "Specific Location": "drop: describes a place, not the squirrel",
    "geo_nearest_place_dist_m": "drop: people only", "geo_nearest_place_direction": "drop: people only",
    "geo_where_text": "drop: people only",
    "date": "drop: day-level (only 11 days)",
    **{c: "drop: day-level (only 11 days)" for c in DAY_LEVEL_NUMERIC + DAY_LEVEL_BINARY},
    TARGET: "target (kept apart)",
}

forgotten = [c for c in work.columns if c not in DECISIONS]
assert not forgotten, f"These columns have no decision yet: {forgotten}"

table = pd.DataFrame({"decision": [DECISIONS[c] for c in work.columns],
                      "% missing": (work.isna().mean() * 100).round(1)}, index=work.columns)
print(f"{table['decision'].str.startswith('keep').sum()} columns kept, "
      f"{table['decision'].str.startswith('drop').sum()} dropped, 1 target")
table

27 columns kept, 21 dropped, 1 target


,decision,% missing
X,keep,0.0
Y,keep,0.0
Unique Squirrel ID,drop: identifier,0.0
Hectare,drop: X and Y say where,0.0
Shift,keep,0.0
Date,drop: duplicate of date,0.0
Hectare Squirrel Number,"drop: volunteer counting order, not known at report time",0.0
Age,"keep, unknown -> 'Unknown'",4.0
Primary Fur Color,"keep, missing -> 'Unknown'",1.7
Highlight Fur Color,"keep, one 0/1 column per color",35.4


## 1. Data cleaning

What was found in the Exploration notebook, and what is done about it:

| Problem found | What is done | Why |
|---|---|---|
| `Age`: 4 values `?` and 97 missing | Both become the category **Unknown** | Not knowing the age is one situation, and it may carry signal. Dropping 101 rows would lose data |
| `Primary Fur Color`: 42 missing (19% diseased, too few to tell) | Category **Unknown** | The strongest feature must not lose rows |
| `Location`: 55 missing | Category **Unknown** | Same reason |
| `Above Ground Sighter Measurement` holds numbers **and the text `FALSE`** (on the ground), and 93 are missing | `FALSE` becomes **0 ft**, text becomes a number, missing stays missing and gets a flag column `height_missing` | A model needs a number, and "unknown height" is different from "on the ground" |
| `Highlight Fur Color`: one cell can hold several colors | Split in section 3 | Needed to read each color |
| Behaviors: 46 sightings with no box ticked, 16 that both approach and run away | **Left as they are** | There is no correct value to put in. They are real volunteer noise |
| Outliers | **None removed** | Coordinates and distances show no outliers. The tallest height (100 ft) is plausible for a tree, and the log in section 3 tames it |
| Duplicate rows | Nothing to do | None found. Three IDs appear twice, but the ID is dropped anyway |
| Any number still missing | Filled with the **median** of the training rows, in the pipeline (section 4) | So a new sighting with a gap can still be scored |

All of this is done by the class `SquirrelFeatures` in `custom_transformers.py`. It never changes the table it receives.

In [5]:
features = SquirrelFeatures().fit(work)         # learns the distance columns and highlight colors from the training rows
cleaned = features.transform(work)

# Before and after for the cleaned columns
print("Age, before:    ", work["Age"].value_counts(dropna=False).to_dict())
print("Age, after:     ", cleaned["Age"].value_counts().to_dict(), "\n")
print("Primary fur, before:", work["Primary Fur Color"].value_counts(dropna=False).to_dict())
print("Primary fur, after: ", cleaned["Primary Fur Color"].value_counts().to_dict(), "\n")
print("Location, before:", work["Location"].value_counts(dropna=False).to_dict())
print("Location, after: ", cleaned["Location"].value_counts().to_dict(), "\n")

# Height: text 'FALSE' -> 0, missing stays missing and is flagged
raw_height = work["Above Ground Sighter Measurement"]
n_false = int(raw_height.astype(str).str.upper().eq("FALSE").sum())
print(f"Height: {n_false} 'FALSE' values, {int(raw_height.isna().sum())} missing")
print(f"        after: {int((cleaned['height_ft'] == 0).sum())} at 0 ft, {int(cleaned['height_missing'].sum())} flagged missing")
assert int(cleaned["height_missing"].sum()) == int(raw_height.isna().sum())
assert int((cleaned["height_ft"] == 0).sum()) >= n_false
assert work.equals(train), "cleaning must not change the table it receives"

Age, before:     {'Adult': 2053, 'Juvenile': 264, nan: 97, '?': 4}
Age, after:      {'Adult': 2053, 'Juvenile': 264, 'Unknown': 101} 

Primary fur, before: {'Gray': 1982, 'Cinnamon': 311, 'Black': 83, nan: 42}
Primary fur, after:  {'Gray': 1982, 'Cinnamon': 311, 'Black': 83, 'Unknown': 42} 

Location, before: {'Ground Plane': 1706, 'Above Ground': 657, nan: 55}
Location, after:  {'Ground Plane': 1706, 'Above Ground': 657, 'Unknown': 55} 

Height: 1706 'FALSE' values, 93 missing
        after: 1706 at 0 ft, 93 flagged missing


## 2. Feature selection

The decisions above drop everything that is an identifier, a duplicate, a place description, people-only, or only a "which day" code. The three notes columns are kept only for the optional keyword similarity setting. That leaves **a small set of features (about 40 columns after encoding)**.

**Weak features were kept on purpose.** `Shift`, `Age` and most behaviors carry almost no signal (their mutual information is close to 0), but with so few columns they cost little, and the regularized models in the next step can ignore them. A quick round of feature selection is part of the Short-List step, when the models can show which ones really never help.

**Check: do the kept numeric columns repeat each other?** Exploration found that `Y`, `hectare_row` and `geo_dist_hill_m` were nearly the same column. The cell below repeats the check on what is kept.

In [6]:
numeric_kept = cleaned.select_dtypes("float").drop(columns=["height_missing"], errors="ignore")
corr = numeric_kept.corr()
pairs = (corr.where(np.triu(np.ones(corr.shape, bool), k=1)).stack().rename("corr").reset_index()
         .rename(columns={"level_0": "column A", "level_1": "column B"}))
strong = pairs[pairs["corr"].abs() >= 0.8].sort_values("corr", key=abs, ascending=False)

print("Kept numeric columns:", list(numeric_kept.columns))
print("Pairs with |correlation| >= 0.8:")
print(strong.round(3).to_string(index=False) if len(strong) else "none")

Kept numeric columns: ['X', 'Y', 'geo_dist_green_m', 'geo_dist_water_m', 'height_ft']
Pairs with |correlation| >= 0.8:
column A column B  corr
       X        Y 0.908


**How to read the result.** `X` (longitude) and `Y` (latitude) are expected to correlate, because Central Park is a long, tilted rectangle. They are the two coordinates of the same point, so both are kept. Any other pair that shows up here would be a candidate to drop, and would be noted in the write-up.

## 3. Feature engineering

**What is built**

| New column(s) | Built from | Why |
|---|---|---|
| `height_ft`, `height_missing` | `Above Ground Sighter Measurement` | Mixed text and numbers become one number plus a flag. Heights are strongly right-skewed (most are 0 ft, a few are very high), so `log(1 + feet)` is used |
| `highlight_black`, `highlight_cinnamon`, `highlight_gray`, `highlight_white` | `Highlight Fur Color` | One cell can hold several colors, so each color gets its own 0/1 column. A color needs at least 10 training rows to get a column |
| `kw_aggression`, `kw_foaming`, `kw_fur`, `kw_ill`, `kw_any`, `has_note` | `Color notes`, `Other Activities`, `Other Interactions` | **Optional** (setting `use_keyword_similarity`, off by default). The **cosine similarity** between the note and a list of symptom keywords (the largest value of each theme), and a flag for "has a note". The keywords come from the symptoms in the Frame step, not from the label. See section 3b |
| `sem_aggression`, `sem_foaming`, `sem_fur`, `sem_ill`, `sem_any`, `has_note` | The same three notes columns | **Optional** (setting `use_spacy_similarity`, off by default). The cosine similarity between the **spaCy word vectors** of the note and short descriptions of the same four symptom themes. It matches by meaning, not by spelling. See section 3c |

**What was considered and not built, with the reason**

| Idea | Decision | Reason |
|---|---|---|
| Hectare row and column | Not built | The row correlates 0.994 with `Y`, so it adds nothing |
| Share diseased among nearby sightings | Not built | The map showed no strong hotspots, and the "diseased in this hectare earlier" rule cost more than never sending anyone. It would also have to be computed inside each cross-validation fold to avoid leaking the label |
| Yes/no flags for single words from free text | Not built | The symptom words are not in the notes, and the common words mostly repeat the tick boxes. The cosine similarity columns above replace them, and they also catch word variants |
| Number of behaviors, "alarm" count | Not built | The share diseased is flat across the number of behaviors ticked |
| Fur color times `Approaches` | Not built | `Approaches` raises the rate within every fur color, so models can add the two effects themselves |
| Date, day of week, weather | Not built | Only 11 distinct days (section 5 lets them back in) |
| Log of the distances | Not by default | In Exploration the distances were not strongly right-skewed (water -0.02, green 0.71, after the islands were removed from the named places), and the log is only worth it for a strong right tail. The setting `log_distances` can test it |
| Discretizing numbers | Not needed | No numeric column needs bins |

In [7]:
no_log = SquirrelFeatures(log_height=False, log_distances=False).fit(work).transform(work)
all_log = SquirrelFeatures(log_height=True, log_distances=True).fit(work).transform(work)
cols = features.distance_columns_ + ["height_ft"]

print("Skew of the numeric columns (0 = symmetric, above 1 = long right tail):")
print(pd.DataFrame({"without log": no_log[cols].skew(), "with log": all_log[cols].skew(),
                    "used": ["log" if c == "height_ft" else "no log" for c in cols]}).round(2).to_string(), "\n")

highlight = cleaned.filter(like="highlight_").sum().astype(int)
print("Sightings per highlight color (a sighting can have several):", highlight.to_dict())

Skew of the numeric columns (0 = symmetric, above 1 = long right tail):
                  without log  with log    used
geo_dist_green_m         0.71     -0.52  no log
geo_dist_water_m        -0.02     -1.64  no log
height_ft                3.92      1.50     log 

Sightings per highlight color (a sighting can have several): {'highlight_black': 75, 'highlight_cinnamon': 878, 'highlight_gray': 177, 'highlight_white': 775}


**Reading the skew table.** The rule is: use a log only for a column with a strong right tail (skew above 1). `height_ft` qualifies, so it gets the log. If a distance column ever shows a skew above 1 here, switch `log_distances` on. The two settings `log_height` and `log_distances` can be tested on and off later.

### 3b. Keyword similarity (optional text features)

**How it works.** The keywords are in `custom_transformers.py` (`KEYWORDS`), in four themes: **aggression**, **foaming**, **fur** (mangy, scabs, injuries) and **ill**. They come from the symptoms in the Frame step (aggression, foaming at the mouth, mangy fur) plus plain synonyms. Words that would match normal text were left out on purpose: `mad` (the census text "made executive adjustments" contains it), `mouth` ("nut in mouth") and `chasing` (already a tick box).

1. The three notes columns are joined into one note and cleaned (lower case, the text the census adds by itself is removed).
2. Note and keywords become vectors of **word stems** (so "aggressively" matches "aggressive", but "curious" does not match "furious"). Rare stems weigh more. These weights are learned in `fit` from the **training notes only**. The label is never used.
3. The **cosine similarity** between the note and each keyword is computed (0 = nothing in common, 1 = the note is that keyword). Each theme keeps its **largest** value.

The columns are already between 0 and 1, so they are **not scaled**. A sighting with no note gets zeros and `has_note = 0`.

The cell below only looks at the notes (no label), to check that the scores behave.

In [8]:
text_features = SquirrelFeatures(use_keyword_similarity=True).fit(work)
kw = text_features.keyword_                       # the fitted KeywordSimilarity (word weights from the training notes)
notes = text_features._notes(work)
sims = kw.similarities(notes)                     # every note against every single keyword
scores = kw.transform(notes)

print("Themes and number of keywords:", {theme: len(words) for theme, words in KEYWORDS.items()})
print(f"Sightings with a note: {int(scores['has_note'].sum()):,} of {len(work):,} ({scores['has_note'].mean():.0%})")
print(f"Sightings matching any keyword: {int((scores['kw_any'] > 0).sum())} (above 0.5: {int((scores['kw_any'] > 0.5).sum())})\n")

best = sims.max(axis=1).sort_values(ascending=False)
best = best[best > 0].head(12)
print("The notes closest to a keyword:")
print(pd.DataFrame({"similarity": best.round(2), "closest keyword": sims.loc[best.index].idxmax(axis=1),
                    "note (cleaned)": kw.clean(notes).loc[best.index].str.slice(0, 70)}).to_string(index=False), "\n")

# Made-up notes, to see how the score behaves
examples = pd.Series(["foaming at the mouth and aggressive", "very aggressively attacking a dog", "nut in mouth",
                      "curious about me", "mangy fur, bald patches",
                      "gray & cinnamon selected as primary. made executive adjustments.", ""])
print(pd.concat([examples.rename("note"), kw.transform(examples).round(2)], axis=1).to_string(index=False))

Themes and number of keywords: {'aggression': 25, 'foaming': 9, 'fur': 18, 'ill': 18}
Sightings with a note: 572 of 2,418 (24%)
Sightings matching any keyword: 11 (above 0.5: 9)

The notes closest to a keyword:
 similarity closest keyword                                          note (cleaned)
       1.00       scratched                                              scratching
       1.00       scratched                                             scratching!
       1.00       scratched                                              scratching
       0.82       scratched                                 chasing (#8),scratching
       0.75       scratched                                  watching me scratching
       0.67           fight                                     fighting w/ pigeons
       0.65       scratched                                     grooming,scratching
       0.62       scratched                                         scratching self
       0.62       scratched      

                                                            note  kw_aggression  kw_foaming  kw_fur  kw_ill  kw_any  has_note
                             foaming at the mouth and aggressive           0.52         0.5     0.0     0.0    0.52         1
                               very aggressively attacking a dog           0.59         0.0     0.0     0.0    0.59         1
                                                    nut in mouth           0.00         0.0     0.0     0.0    0.00         1
                                                curious about me           0.00         0.0     0.0     0.0    0.00         1
                                         mangy fur, bald patches           0.00         0.0     0.5     0.0    0.50         1
gray & cinnamon selected as primary. made executive adjustments.           0.00         0.0     0.0     0.0    0.00         0
                                                                           0.00         0.0     0.0     0.0    0.00   

**Reading the result.**
- **Almost every score is 0.** The real symptom words (foaming, mangy, aggressive) are not in the notes, so only a few notes match, and those are mostly everyday words close to a keyword (for example "scratching"). The columns are therefore rare, and a model has very little to learn from.
- The made-up notes show what the score does when the words *are* there: "foaming at the mouth and aggressive" scores about 0.5 on `foaming` and on `aggression` (a longer note scores lower than a single word), and "nut in mouth", "curious" and the census boilerplate score 0.
- **Does it help the model?** That is not decided here. The Training step (section 6c) tests it with cross-validation, with and without the `has_note` flag, and the setting stays **off** unless it helps.
- **A caution.** The notes only help if the reporting app collects the same free-text notes. Otherwise every real report looks like "no note".

### 3c. spaCy meaning similarity (optional text features, from the Text Processing notebook)

The keyword method in 3b only fires when a note shares a **word stem** with a keyword. So "hostile, lunged at my dog" scores 0, because no keyword is spelled like those words. The Text Processing class notebook showed another way: **word vectors**.

**How it works** (`SpacyConceptSimilarity` in `custom_transformers.py`)
1. spaCy's `en_core_web_lg` model gives most English words a list of 300 numbers (a **vector**). Words used in similar ways ("hostile", "aggressive", "angry") get similar vectors.
2. A note's vector is the **average** of its word vectors. **Stop words** ("the", "and", "at") and punctuation are left out, because they would pull every short note towards the same "plain English" average. The census boilerplate is removed first, as in 3b.
3. Each of the four symptom themes is described by two short phrases (`CONCEPTS`), for example "foaming at the mouth" and "drooling saliva". They describe the symptoms from the Frame step and were **not** picked by looking at the label. The word "squirrel" is left out, because it is in many notes and would make every note look a little similar.
4. The **cosine similarity** between the note and each phrase is computed, and each theme keeps its largest value (`sem_<theme>`), plus `sem_any` (the largest of all) and `has_note`.

**What is learned in `fit`?** Nothing from the data: the phrase vectors come from spaCy, and the model never sees the label. So this step cannot leak anything from the validation folds or the test set.

**Difference from the keywords.** Every note gets a score (usually 0.2 to 0.6), not only the ones with a keyword. That is more information, but also more noise. The cell below compares the two methods on the training notes (no label used).

In [9]:
spacy_features = SquirrelFeatures(use_spacy_similarity=True).fit(work)
sem = spacy_features.spacy_                           # the fitted SpacyConceptSimilarity
sem_scores = sem.transform(notes)                     # same joined notes as in 3b
has = sem_scores["has_note"] == 1

print("Concept phrases:", CONCEPTS, "\n")
print(f"Sightings with a note: {int(has.sum()):,}. Their spaCy scores (sightings without a note get 0):")
print(sem_scores.loc[has].drop(columns="has_note").describe().round(2).T[["mean", "25%", "50%", "75%", "max"]].to_string(), "\n")

# The notes closest to each theme in meaning
cleaned_notes = KeywordSimilarity.clean(notes)
for theme in sem.themes_:
    top = sem_scores[f"sem_{theme}"].sort_values(ascending=False).head(4)
    print(f"Closest to '{theme}':", " | ".join(f"{cleaned_notes[i][:45]!r} ({v:.2f})" for i, v in top.items()))

# Do the two methods agree? (only notes, no label)
both = pd.DataFrame({"keyword kw_any": scores.loc[has, "kw_any"], "spaCy sem_any": sem_scores.loc[has, "sem_any"]})
print(f"\nCorrelation between kw_any and sem_any over the notes: {both.corr().iloc[0, 1]:.2f}")

# Made-up notes: four that describe a symptom WITHOUT any word from the keyword lists, then the ones from 3b
examples_sem = pd.Series(["snapped at people and would not back off", "wobbly, slow and shaking", "spit everywhere, wet chin",
                          "fur missing in clumps", "foaming at the mouth and aggressive", "nut in mouth",
                          "curious about me", "mangy fur, bald patches",
                          "gray & cinnamon selected as primary. made executive adjustments.", ""])
print()
print(pd.concat([examples_sem.rename("note"), kw.transform(examples_sem)["kw_any"].round(2),
                 sem.transform(examples_sem).drop(columns="has_note").round(2)], axis=1).to_string(index=False))

Concept phrases: {'aggression': ['aggressive attacking and biting', 'angry hostile charging at people'], 'foaming': ['foaming at the mouth', 'drooling saliva'], 'fur': ['mangy fur with bald patches', 'scabs wounds and injured skin'], 'ill': ['sick weak and lethargic', 'stumbling confused erratic movement']} 

Sightings with a note: 572. Their spaCy scores (sightings without a note get 0):
                mean   25%   50%   75%   max
sem_aggression  0.41  0.36  0.42  0.48  0.74
sem_foaming     0.32  0.26  0.32  0.39  0.71
sem_fur         0.34  0.28  0.34  0.42  0.71
sem_ill         0.39  0.33  0.39  0.46  0.71
sem_any         0.45  0.39  0.44  0.51  0.74 

Closest to 'aggression': 'avoided people' (0.74) | 'avoided people' (0.74) | 'avoided people' (0.74) | 'wandering,interacting with tourists approache' (0.65)
Closest to 'foaming': 'nut in mouth' (0.71) | 'nut in mouth' (0.71) | 'nut in mouth' (0.71) | 'nut in mouth' (0.71)
Closest to 'fur': 'patches of white on haunches' (0.71) | 'gra

**Reading the result.**
- **It catches meaning, not spelling.** The first four made-up notes ("snapped at people and would not back off", "wobbly, slow and shaking", "spit everywhere, wet chin", "fur missing in clumps") share no word with the keyword lists, so `kw_any` is 0. The spaCy scores put each one highest on the right theme (aggression, ill, foaming, fur).
- **It is also fooled by averages.** "nut in mouth" scores about 0.7 on `foaming`, because "mouth" pulls the average towards "foaming at the mouth". The keyword method was built to avoid exactly that. An averaged vector keeps the topic of a note but loses its details.
- **On the real notes it is mostly fooled.** The real notes are about ordinary things, so the closest ones are false alarms: "avoided people" is closest to `aggression` (the vector of "people" is shared with "charging at people"), "nut in mouth" to `foaming`, and fur color notes such as "patches of white on haunches" to `fur`. The two methods hardly agree (correlation about 0.1). As in 3b, the real symptom words are not in the census.
- **Does it help the model?** Decided in the Training step (section 6c), against the keyword method and against the "has a note" flag alone, with the same cross-validation rule. The setting stays **off** unless it helps.
- **To use it later** the reporting app must collect free-text notes, and the computer that runs the model needs spaCy and `en_core_web_lg` installed.

## 4. Feature scaling and encoding

| Kind of column | What is done | Why |
|---|---|---|
| **Numbers** (`X`, `Y`, distances, `height_ft`) | Fill any gap with the median, then **standardize** (mean 0, standard deviation 1) | Linear models, SVMs and neural nets are sensitive to the size of the numbers. Trees do not mind, so it is harmless for them |
| **Categories** (`Primary Fur Color`, `Age`, `Location`, `Shift`, `geo_nearest_place_kind`) | **One-hot**: one 0/1 column per value | Models need numbers. Values seen in fewer than 5 training rows are grouped as "infrequent", and a value never seen in training becomes all zeros instead of crashing |
| **0/1 columns** (behaviors, highlight colors, `height_missing`) | Left as they are | Already on the same scale |

The medians, means, standard deviations and categories are all learned from the **training set** inside `fit`. When the preprocessor is later used on the test set, it only applies what it learned.

In [10]:
preprocessor = make_preprocessor()
X_train = preprocessor.fit_transform(work)        # fitted on the training copy only

# The result must be ready to train on
assert X_train.shape[0] == len(y_train) and X_train.index.equals(work.index)
assert not X_train.isna().any().any(), "there are still missing values"
assert all(pd.api.types.is_numeric_dtype(t) for t in X_train.dtypes), "all columns must be numbers"
assert TARGET not in X_train.columns, "the label must never be a feature"

print(f"Prepared training table: {X_train.shape[0]:,} rows x {X_train.shape[1]} columns")
numbers = ["X", "Y", "height_ft"] + features.distance_columns_
print("Standardized columns (mean should be 0 and std 1):")
print(X_train[numbers].agg(["mean", "std"]).T.round(3).to_string(), "\n")
print("All columns:", list(X_train.columns))

Prepared training table: 2,418 rows x 39 columns
Standardized columns (mean should be 0 and std 1):
                  mean  std
X                  0.0  1.0
Y                 -0.0  1.0
height_ft          0.0  1.0
geo_dist_green_m   0.0  1.0
geo_dist_water_m  -0.0  1.0 

All columns: ['X', 'Y', 'geo_dist_green_m', 'geo_dist_water_m', 'height_ft', 'Primary Fur Color_Black', 'Primary Fur Color_Cinnamon', 'Primary Fur Color_Gray', 'Primary Fur Color_Unknown', 'Age_Adult', 'Age_Juvenile', 'Age_Unknown', 'Location_Above Ground', 'Location_Ground Plane', 'Location_Unknown', 'Shift_AM', 'Shift_PM', 'geo_nearest_place_kind_green', 'geo_nearest_place_kind_hill', 'geo_nearest_place_kind_none', 'geo_nearest_place_kind_water', 'height_missing', 'Running', 'Chasing', 'Climbing', 'Eating', 'Foraging', 'Kuks', 'Quaas', 'Moans', 'Tail flags', 'Tail twitches', 'Approaches', 'Indifferent', 'Runs from', 'highlight_black', 'highlight_cinnamon', 'highlight_gray', 'highlight_white']


## 5. Settings that can be tuned later

The recipe is built by one function, `make_preprocessor(...)`. Each choice made above is a **setting**, so the Fine-Tune step can treat the data preparation as hyperparameters (as the checklist suggests when you are not sure about a choice).

| Setting | Default | What it changes |
|---|---|---|
| `keep_day_level` | `False` | Bring back the weather and calendar columns (and a flag for missing wind) |
| `drop_hill_distance` | `True` | Keep `geo_dist_hill_m` too (it repeats `Y`) |
| `log_height` | `True` | Use `log(1 + feet)` for the height above ground |
| `log_distances` | `False` | Use `log(1 + metres)` for the distances to named places |
| `use_nearest_place_kind` | `True` | Keep the kind of the nearest named place |
| `use_highlight_colors` | `True` | Add the highlight color columns |
| `min_highlight_count` | `10` | How many training rows a highlight color needs to get a column |
| `use_keyword_similarity` | `False` | Add the keyword similarity columns (`kw_*`) and the `has_note` flag. Needs the three notes columns |
| `use_spacy_similarity` | `False` | Add the spaCy meaning similarity columns (`sem_*`) and the `has_note` flag. Needs the three notes columns and spaCy |
| `scaler` | `'standard'` | `'standard'`, `'robust'` (less sensitive to outliers) or `'none'` |
| `impute_strategy` | `'median'` | Fill gaps with the `'median'` or the `'mean'` |
| `min_category_count` | `5` | Categories rarer than this are grouped as "infrequent" |

Because the settings are ordinary parameters, they can be searched with scikit-learn (for example `features__keep_day_level` or `columns__numbers__scale__with_mean`). The cell below checks that every setting builds a working recipe.

In [11]:
variants = {
    "default": {},
    "bring back weather and calendar": {"keep_day_level": True},
    "keep the hill distance": {"drop_hill_distance": False},
    "no log on the height": {"log_height": False},
    "log the distances": {"log_distances": True},
    "no nearest-place kind": {"use_nearest_place_kind": False},
    "no highlight colors": {"use_highlight_colors": False},
    "keyword similarity of the notes": {"use_keyword_similarity": True},
    "spaCy similarity of the notes": {"use_spacy_similarity": True},
    "robust scaler": {"scaler": "robust"},
    "no scaling": {"scaler": "none"},
    "mean instead of median": {"impute_strategy": "mean"},
    "group rare categories (< 50)": {"min_category_count": 50, "min_highlight_count": 100},
}
rows = []
for name, settings in variants.items():
    X_variant = make_preprocessor(**settings).fit_transform(work)
    assert not X_variant.isna().any().any()
    rows.append({"variant": name, "columns": X_variant.shape[1]})
pd.DataFrame(rows).set_index("variant")

,columns
variant,
default,39
bring back weather and calendar,48
keep the hill distance,40
no log on the height,39
log the distances,39
no nearest-place kind,35
no highlight colors,35
keyword similarity of the notes,45
spaCy similarity of the notes,45


## 6. Checks

**Does the recipe work on a table it was not fitted on?** It must, because the test set and new sightings will go through it. To check this **without opening the test set**, the training copy is split in two: the recipe is fitted on one part and applied to the other, with the label column removed (as it will be for a new sighting).

The test also covers three things that happen in real use: a fur color never seen in training, missing values, and a missing column (which must give a clear error instead of a silent wrong answer).

In [12]:
# Fit on one part of the training copy, apply to the other part (the test set is not used)
part = np.random.default_rng(0).random(len(work)) < 0.8
fit_part, other_part = work[part], work[~part].drop(columns=[TARGET])

for label, settings in [("default", {}), ("tuned (section 7)", TUNED_SETTINGS)]:
    check = make_preprocessor(**settings).fit(fit_part)
    X_other = check.transform(other_part)
    assert list(X_other.columns) == list(check.transform(fit_part).columns), "different columns for different tables"
    assert not X_other.isna().any().any()
    print(f"A table it was not fitted on works, {label} settings:", X_other.shape)

# A new sighting with a fur color never seen, a missing age and a missing height
odd = other_part.head(5).copy()
odd["Primary Fur Color"], odd["Age"], odd["Above Ground Sighter Measurement"] = "Purple", np.nan, np.nan
assert not check.transform(odd).isna().any().any()
print("Unseen and missing values work")

# A missing column gives a clear error
try:
    check.transform(other_part.drop(columns=["Age"]))
except KeyError as err:
    print("Missing column is caught:", err)

A table it was not fitted on works, default settings: (492, 39)
A table it was not fitted on works, tuned (section 7) settings: (492, 46)
Unseen and missing values work
Missing column is caught: "These columns are missing from the table: ['Age']"


## 7. Settings chosen in the Training step, and saving `preprocessor.pkl`

The Training notebook tuned the preparation settings together with the models (sections 8 and 9: random searches over the 11 settings of section 5). As the project asks, what was learned there is **implemented here**: the settings of the final model are written in `custom_transformers.py` as `TUNED_SETTINGS`, and `preprocessor.pkl` is built from them.

| Setting | Default | Tuned | What the change means |
|---|---|---|---|
| `log_distances` | off | **on** | log(1 + metres) for the distances to named places |
| `use_highlight_colors` | on | **off** | the highlight colors mostly repeat the primary color (Exploration) |
| `use_keyword_similarity` | off | **on** | keyword text features of the notes (3b) |
| `use_spacy_similarity` | off | **on** | spaCy text features of the notes (3c) |
| `scaler` | standard | **none** | no scaling: in the search it scored no worse than scaling, and the tree members do not need it |
| `min_category_count` | 5 | **14** | rare categories are grouped a little more |

The other settings are the same as the defaults (`min_highlight_count` also changed, to 46, but it has no effect with the highlight colors off). The cell checks that the result matches the first step of `model.pkl` exactly (when `model.pkl` exists), so the two files cannot drift apart.

In [13]:
final_preprocessor = make_preprocessor(**TUNED_SETTINGS).fit(work)          # fitted on the training copy only
X_final = final_preprocessor.transform(work)
assert not X_final.isna().any().any() and TARGET not in X_final.columns
print("Tuned settings:", TUNED_SETTINGS)
print(f"Prepared training table with the tuned settings: {X_final.shape[0]:,} rows x {X_final.shape[1]} columns")

if Path("model.pkl").exists():                                  # written by the Training notebook
    model_prep = joblib.load("model.pkl").named_steps["prep"]
    same = np.allclose(model_prep.transform(work).to_numpy(float), X_final.to_numpy(float))
    assert same, "the tuned settings no longer match model.pkl: update TUNED_SETTINGS"
    print("Matches the first step of model.pkl exactly")

joblib.dump(final_preprocessor, "preprocessor.pkl")
reloaded = joblib.load("preprocessor.pkl")
sample = train.head(300)                       # a copy of the first training rows, still with the label column
assert reloaded.transform(sample).equals(final_preprocessor.transform(sample))
print(f"preprocessor.pkl saved ({Path('preprocessor.pkl').stat().st_size / 1e3:.0f} KB) and checked after reloading")

Tuned settings: {'keep_day_level': False, 'drop_hill_distance': True, 'log_height': True, 'log_distances': True, 'use_nearest_place_kind': True, 'use_highlight_colors': False, 'min_highlight_count': 46, 'use_keyword_similarity': True, 'use_spacy_similarity': True, 'scaler': 'none', 'impute_strategy': 'median', 'min_category_count': 14}
Prepared training table with the tuned settings: 2,418 rows x 46 columns


Matches the first step of model.pkl exactly
preprocessor.pkl saved (31 KB) and checked after reloading


## Summary of the Prepare step

**What was done**
- Every raw column got a decision and a reason (section "A decision for every column"), checked in code so none is forgotten.
- **Cleaning:** unknown age, fur color and location became their own category. The mixed text and numbers in the height column became a number plus a missing flag. Nothing was dropped as an outlier.
- **Selection:** identifiers, duplicates, free text, people-only columns, the hectare columns and the day-level columns were dropped, with the evidence from Exploration. Weak features were kept for the Short-List step to judge.
- **Engineering:** `height_ft` and `height_missing`, one column per highlight color, and log of the height. Ideas that the data did not support (hectare parts, nearby-disease rate, keyword flags, behavior counts) were written down with the reason.
- **Text:** an optional setting, `use_keyword_similarity`, adds the cosine similarity between the notes and symptom keywords (4 themes) and a "has a note" flag. A second setting, `use_spacy_similarity`, does the same with **spaCy word vectors** (the method from the Text Processing notebook), which matches notes by meaning instead of spelling. Both are off by default, and the Training step tests whether either helps.
- **Scaling and encoding:** median fill and standardization for numbers, one-hot for categories, with rare and unseen values handled.
- **Tuned settings (section 7):** the Training step tuned these settings with the models; the final ones (`TUNED_SETTINGS`: both text features on, log of the distances, no highlight colors, no scaling) are implemented here.
- **Saved:** `preprocessor.pkl`, built with the tuned settings and checked against `model.pkl`. No prepared data was saved or committed.

**How the Training notebook should use it**
- To **fit and tune**, put a fresh `make_preprocessor(...)` **inside** the model pipeline, so it is refitted on each cross-validation fold: `Pipeline([("prep", make_preprocessor()), ("model", ...)])`. Preparing the whole training set first and then cross-validating would leak the medians and means.
- Keep `custom_transformers.py` next to `preprocessor.pkl`, because loading the pickle needs it.

**Things to remember**
- Weather and calendar are **off** by default. Turning them on is a setting to test with cross-validation that holds out whole days, because the random split shares days between train and test.
- The random split can make day-level features look better than they are. The day-grouped cross-validation promised in the Get the Data step is the check for that.
- The recipe only uses columns that exist when a report arrives. It does not use the ID or the squirrel number. It uses the notes **only** if `use_keyword_similarity` or `use_spacy_similarity` is on, and then the reporting app must collect the same notes (an empty note is fine).